# Palyginimas ir rezultatai

Šis notebook **nemoko** modelių. Jis skaito keturis failus, kuriuos atsisiuntėte po 01–04 paleidimo Colab'e:

- `preds_baseline.csv`
- `preds_catboost.csv`
- `preds_svm.csv`
- `preds_mlp.csv`

Colab: kairėje **Files → Upload** ir įkelkite visus 4 failus į šią sesiją (prieš paleisdami celes).

## 0. Bibliotekos

Čia tik metrikos ir grafikai — CatBoost paketo nereikia.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
)
from sklearn.calibration import calibration_curve


## 1. Kur ieškoti failų

Ieškoma čia pat, šiame Colab aplanke (ten, kur įkėlėte `preds_*.csv`), ir jei reikia — poaplankyje `outputs/`.

In [ ]:
SEED = 42
C_CALL = 1.0
V_SUCCESS = 10.0
K_FRACTION = 0.10

EXPECTED = {
    "baseline": "preds_baseline.csv",
    "catboost": "preds_catboost.csv",
    "svm": "preds_svm.csv",
    "mlp": "preds_mlp.csv",
}
LABELS = {
    "baseline": "Logistinė regresija (baseline 2)",
    "catboost": "CatBoost (pagrindinis)",
    "svm": "SVM (RBF + Platt)",
    "mlp": "MLP",
}

def find_pred_file(fname):
    # Pirma šis aplankas (Colab Upload), tada outputs/.
    for p in [Path(fname), Path("outputs") / fname]:
        if p.exists():
            return p
    return None

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Darbinis aplankas:", ROOT)


## 2. Keturių `preds_*.csv` įkėlimas

Kiekvienas failas turi stulpelius `y` (tikras atsakymas teste) ir `p` (modelio $\hat p$). Jei kurio nors trūksta, žemiau parašys **kurio tiksliai** ir kurį notebook'ą paleisti.

In [ ]:
preds = {}
missing = []
for key, fname in EXPECTED.items():
    path = find_pred_file(fname)
    if path is None:
        missing.append((key, fname))
        continue
    tab = pd.read_csv(path)
    if not {"y", "p"} <= set(tab.columns):
        raise ValueError(f"{path} turi stulpelius {list(tab.columns)}, reikia y ir p")
    y = tab["y"].to_numpy().astype(int)
    p = tab["p"].to_numpy().astype(float)
    preds[key] = (y, p)
    print(f"OK  {key:10s}  {path.name:22s}  n={len(y)}  y=1 dažnis={y.mean():.4f}")

if missing:
    print("\nTRŪKSTA failų:")
    for key, fname in missing:
        nb = {
            "baseline": "01_baseline_logistine_regresija.ipynb",
            "catboost": "02_catboost.ipynb",
            "svm": "03_svm.ipynb",
            "mlp": "04_mlp.ipynb",
        }[key]
        print(f"  - {fname}  (modelis: {LABELS[key]})  → paleiskite {nb}")
    print("Palyginimas bus tik iš tų modelių, kurių failai yra.")
else:
    print("\nVisi 4 failai rasti.")

if not preds:
    raise FileNotFoundError(
        "Nerasta nė vieno preds_*.csv. Colab: Files → Upload ir įkelkite visus 4 failus."
    )

# Jei visų n sutampa — gerai; jei ne, vis tiek skaičiuojame atskirai, bet grafike įspėjame.
ns = {k: len(v[0]) for k, v in preds.items()}
if len(set(ns.values())) > 1:
    print("Dėmesio: skirtingas eilučių skaičius", ns)


## 3. Bendra rezultatų lentelė

Tos pačios metrikos visiems: **PR-AUC**, **precision@k** (k = 10 % test imties — kiek pataikytume, jei operatorius paskambintų tik geriausiam dešimtadaliui), **Brier** (kuo mažesnis, tuo p̂ arčiau realybės), **kaštai** pagal top-k politiką $c_{call}\cdot k - v_{success}\cdot TP_{top\,k}$ (skambiname tik top-k, ne pagal 0,5 slenkstį), ir **sutikimo dažnis** teste.

In [ ]:
def precision_at_k(y, p, k):
    k = int(min(k, len(y)))
    order = np.argsort(-np.asarray(p), kind="mergesort")
    return float(np.asarray(y)[order][:k].mean()), k

def contact_cost_topk(y, p, k, c_call=C_CALL, v_success=V_SUCCESS):
    k = int(min(k, len(y)))
    order = np.argsort(-np.asarray(p), kind="mergesort")
    tp = int(np.asarray(y)[order][:k].sum())
    return float(c_call * k - v_success * tp), tp

rows = []
curves = {}
for key, (y, p) in preds.items():
    k = max(1, int(K_FRACTION * len(y)))
    pk, k_used = precision_at_k(y, p, k)
    cost, tp = contact_cost_topk(y, p, k)
    rows.append({
        "modelis": LABELS[key],
        "kodas": key,
        "PR-AUC": float(average_precision_score(y, p)),
        "precision@k": pk,
        "k": k_used,
        "Brier": float(brier_score_loss(y, p)),
        "kaštai (top-k)": cost,
        "TP top-k": tp,
        "sutikimo dažnis": float(np.mean(y)),
        "n": int(len(y)),
    })
    curves[LABELS[key]] = (y, p)

table = pd.DataFrame(rows).set_index("kodas")
table_show = table.copy()
for c in ["PR-AUC", "precision@k", "Brier", "sutikimo dažnis"]:
    table_show[c] = table_show[c].map(lambda x: f"{x:.4f}")
table_show["kaštai (top-k)"] = table_show["kaštai (top-k)"].map(lambda x: f"{x:.1f}")
display(table_show[["modelis", "PR-AUC", "precision@k", "k", "Brier", "kaštai (top-k)", "sutikimo dažnis", "n"]])
table.to_csv(OUTPUT_DIR / "rezultatu_lentele.csv", index=True)
print("Lentelė išsaugota:", OUTPUT_DIR / "rezultatu_lentele.csv")
print("Mažesnis Brier ir mažesni kaštai (labiau neigiami = daugiau pelno) — geriau. Didesnis PR-AUC ir precision@k — geriau.")


## 4. Precision–Recall kreivės (visi modeliai vienoje diagramoje)

X = atgaminimas (kiek tikrų „taip“ pagavome), Y = tikslumas (kokia dalis skambučių pataikė). Kreivė aukščiau ir dešiniau — geriau, kai klasių disbalansas.

In [ ]:
plt.figure(figsize=(8, 5))
for name, (y, p) in curves.items():
    prec, rec, _ = precision_recall_curve(y, p)
    ap = average_precision_score(y, p)
    plt.plot(rec, prec, label=f"{name} (PR-AUC={ap:.3f})")
plt.xlabel("Atgaminimas (Recall)")
plt.ylabel("Tikslumas (Precision)")
plt.title("Precision–Recall: visi turimi modeliai")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pr_all_models.png", dpi=140)
plt.show()


## 5. Kalibracijos grafikas (visi modeliai vienoje diagramoje)

Jei kreivė arti įstrižainės, $\hat p=0.7$ tikrai reiškia ~70 % sutarčių. Virš įstrižainės — modelis per atsargus; po įstrižaine — per drąsus (per didelės tikimybės).

In [ ]:
plt.figure(figsize=(8, 5))
for name, (y, p) in curves.items():
    frac, mean_p = calibration_curve(y, p, n_bins=10, strategy="quantile")
    br = brier_score_loss(y, p)
    plt.plot(mean_p, frac, marker="o", label=f"{name} (Brier={br:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="ideali kalibracija")
plt.xlabel("Vidutinė prognozuota p̂")
plt.ylabel("Stebėta teigiamų dalis")
plt.title("Kalibracija: visi turimi modeliai")
plt.legend(loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "calibration_all_models.png", dpi=140)
plt.show()


## 6. FN pavyzdžiai (12 klaidingai neigiamų)

FN čia: klientas **tikrai sutiko** (`y=1`), bet pagal top-k politiką **nebūtų paskambinta** (jo $\hat p$ per žema, nepatenka į geriausiųjų 10 %). Tai brangiausios klaidos dokumente.

Požymius paimame iš `bank-full.csv` **tik lentelei** — to paties chronologinio test gabalo eilės tvarka kaip 01–04. Modelių čia nemokome.

In [ ]:
def test_frame_from_csv():
    csv_path = ROOT / "bank-full.csv"
    if not csv_path.exists():
        return None
    df = pd.read_csv(csv_path, sep=";")
    months = df["month"].to_numpy()
    ch = [0]
    for i in range(1, len(months)):
        if months[i] != months[i - 1]:
            ch.append(i)
    ch.append(len(df))
    def nearest(t):
        return min(ch, key=lambda c: abs(c - t))
    n = len(df)
    val_end = nearest(int(0.85 * n))
    te = df.iloc[val_end:].copy()
    te["y_bin"] = (te["y"].astype(str).str.lower() == "yes").astype(int)
    return te

te = test_frame_from_csv()
fn_model = "catboost" if "catboost" in preds else max(preds, key=lambda k: average_precision_score(*preds[k]))
y_fn, p_fn = preds[fn_model]
k_fn = max(1, int(K_FRACTION * len(y_fn)))
order = np.argsort(-p_fn, kind="mergesort")
in_topk = np.zeros(len(y_fn), dtype=bool)
in_topk[order[:k_fn]] = True
fn_mask = (y_fn == 1) & (~in_topk)

print(f"FN modelis: {LABELS[fn_model]}")
print(f"Teigiamų teste: {(y_fn==1).sum()}, iš jų FN (neskambintume): {fn_mask.sum()}")

if te is None:
    print("bank-full.csv nerastas — FN lentelė tik su y ir p.")
    fn_tab = pd.DataFrame({"eilutes_pozicija_teste": np.where(fn_mask)[0], "p_hat": p_fn[fn_mask], "y": y_fn[fn_mask]})
    fn_tab = fn_tab.sort_values("p_hat").head(12)
else:
    if len(te) != len(y_fn):
        print(f"Įspėjimas: test n={len(te)}, preds n={len(y_fn)}. Jungiame iki min ilgio.")
    n = min(len(te), len(y_fn))
    te_n = te.iloc[:n].copy()
    y_chk = y_fn[:n]
    if not np.array_equal(te_n["y_bin"].to_numpy(), y_chk):
        print("Įspėjimas: y iš preds nesutampa su bank-full test etiketėmis. Lentelė vis tiek pagal preds eilę.")
    te_n = te_n.iloc[np.where(fn_mask[:n])[0]].copy()
    te_n["p_hat"] = p_fn[:n][fn_mask[:n]]
    te_n["eilutes_nr"] = te_n.index.astype(int)
    cols = [c for c in ["eilutes_nr","age","job","marital","education","balance","housing","loan",
                        "contact","month","campaign","poutcome","p_hat"] if c in te_n.columns]
    fn_tab = te_n[cols].sort_values("p_hat").head(12)

display(fn_tab.reset_index(drop=True))
fn_tab.to_csv(OUTPUT_DIR / f"fn_examples_{fn_model}_palyginimas.csv", index=False)
print("Tai klientai, kurie sutiko, bet modelis jiems davė mažiausią p̂ (labiausiai „netikėti“ pirkėjai).")


## 7. Hipotezė ir concept drift (kodėl skaičiai tokie)

**Hipotezės kriterijus (koliokviumo dokumentas):** CatBoost laimi prieš logistinę regresiją, jei teste, be `duration`, **PR-AUC skirtumas ≥ 0,03 IR bootstrap p < 0,05** (1000 pakartojimų).

**Kodėl visų modelių aukščiausios p̂ dažnai „nemuša“ realaus sutikimo:** tai ne atskiro modelio klaida. Chronologinis skaidymas mokymą palieka ankstyvose kampanijose (sutikimo dažnis traine ~**5,8 %**), o testą — vėlyvosiose (~**45,5 %**). Tas pats `poutcome=success` traine reiškia ~19 % sutikimo, teste ~73 %. Dokumento 4.5 lentelė tai vadina **sąlygų pasikeitimu (concept drift)**: 2008–2010 kampanijos keitėsi, todėl ankstyvų duomenų taisyklės (ypač retas `month=oct` su 80 eilučių) vėliau nebegalioja. Todėl visos architektūros pervertina p̂ aukščiausiame segmente, o PR-AUC teste būna arti bazinio dažnio.

Kitoje celėje skaičiai imami **iš jūsų `preds_*.csv`**, todėl verdiktas visada atitinka šį paleidimą.

In [ ]:
def bootstrap_diff(y, p1, p0, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs[i] = average_precision_score(y[idx], p1[idx]) - average_precision_score(y[idx], p0[idx])
    mean = float(diffs.mean())
    p = float(2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return mean, min(p, 1.0), (float(lo), float(hi))

best_key = max(preds, key=lambda k: average_precision_score(*preds[k]))
best_ap = average_precision_score(*preds[best_key])
lines = [
    f"**Geriausias pagal PR-AUC šiame paleidime:** {LABELS[best_key]} (PR-AUC = {best_ap:.4f}).",
    "",
]
if "catboost" in preds and "baseline" in preds:
    y_a, p_cb = preds["catboost"]
    y_b, p_lr = preds["baseline"]
    n = min(len(y_a), len(y_b))
    if not np.array_equal(y_a[:n], y_b[:n]):
        lines.append("Įspėjimas: CatBoost ir LogReg `y` vektoriai nesutampa — bootstrap vis tiek ant bendro ilgio, bet patikrinkite, ar 01 ir 02 naudoto tas pats testas.")
    y_c = y_a[:n]
    delta, pval, ci = bootstrap_diff(y_c, p_cb[:n], p_lr[:n], n_boot=1000, seed=SEED)
    ap_cb = average_precision_score(y_c, p_cb[:n])
    ap_lr = average_precision_score(y_c, p_lr[:n])
    win = (delta >= 0.03) and (pval < 0.05)
    lines += [
        f"CatBoost PR-AUC = {ap_cb:.4f}; logistinė regresija = {ap_lr:.4f}; Δ = {ap_cb - ap_lr:.4f} (reikia ≥ 0,03).",
        f"Bootstrap (1000): Δ vidurkis = {delta:.4f}, 95% PI [{ci[0]:.4f}, {ci[1]:.4f}], p = {pval:.4f}.",
        "**Hipotezė PRIIMAMA.**" if win else "**Hipotezė ATMETAMA.**",
        "",
        "Jei hipotezė atmesta: CatBoost neįrodė statistiškai reikšmingo +0,03 PR-AUC prieš tiesinį baseline. "
        "Pagrindinė priežastis — concept drift (žr. markdown aukščiau), ne tai, kad medžiai „sugedo“. "
        "Teste rikiavimas visiems modeliams beveik subyra, todėl sudėtingesnis metodas neturi erdvės laimėti.",
    ]
else:
    need = []
    if "catboost" not in preds:
        need.append("preds_catboost.csv (02_catboost.ipynb)")
    if "baseline" not in preds:
        need.append("preds_baseline.csv (01_baseline_logistine_regresija.ipynb)")
    lines.append("Hipotezės testo negalima skaičiuoti — trūksta: " + "; ".join(need) + ".")

display(Markdown("\n".join(lines)))


Paleidus šią celę, į jūsų kompiuterio Atsisiuntimų (Downloads) aplanką atsisiųs ZIP failas su palyginimo rezultatais: pr_all_models.png, calibration_all_models.png, rezultatu_lentele.csv ir FN lentele. Jį išpakuokite ir PNG įkelkite į dokumentacijos 1 ir 2 pav. dėžutes.

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

# Grafikai ir lentelė yra aplanke outputs/ (sukuriami šiame notebook'e aukščiau).
out = Path("outputs")
out.mkdir(exist_ok=True)
print("Kas bus ZIP viduje:")
for p in sorted(out.glob("*")):
    print(" ", p.name)

shutil.make_archive("palyginimas_outputs", "zip", "outputs")
files.download("palyginimas_outputs.zip")
